# EXPL_00 — Design, Scope, and Frozen Model Contract

This notebook verifies the final explainability contract before any model
interpretation is calculated. No model is changed or retrained.


In [1]:
# Import libraries
from pathlib import Path
import sys
import importlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

In [2]:
# Define config paths
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "explainability.yaml"
CONFIG_PATH

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/configs/explainability.yaml')

In [3]:
# Import modules for generating explainability
from src.ontario_peak_risk.explainability.common import (
    load_explainability_config,
    ensure_phase_directories,
    load_final_metadata,
)

In [4]:
# Load parameters
config, project_root = load_explainability_config(CONFIG_PATH)
phase_paths = ensure_phase_directories(config, project_root)
metadata = load_final_metadata(config, project_root)

print("Project root:", project_root)
print("RF:", metadata["rf"]["algorithm"], "| horizons:", metadata["rf"]["horizons"])
print("XGB:", metadata["xgb"]["algorithm"], "| horizons:", metadata["xgb"]["horizons"])
print("Explainability output:", phase_paths["outputs_dir"])


Project root: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk
RF: RandomForestRegressor | horizons: 24
XGB: XGBoostClassifier | horizons: 24
Explainability output: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\outputs\explainability


In [5]:
# Load contract
rf_features = metadata["rf"]["numeric_features"] + metadata["rf"]["categorical_features"]
xgb_features = metadata["xgb"]["numeric_features"] + metadata["xgb"]["categorical_features"]

contract = pd.DataFrame({
    "feature": rf_features,
    "rf_used": [f in rf_features for f in rf_features],
    "xgb_used": [f in xgb_features for f in rf_features],
})

display(contract)



,feature,rf_used,xgb_used
0,target_hour,True,True
1,target_weekday,True,True
2,target_month,True,True
3,target_is_weekend,True,True
4,target_hour_sin,True,True
5,target_hour_cos,True,True
6,target_weekday_sin,True,True
7,target_weekday_cos,True,True
8,target_month_sin,True,True
9,target_month_cos,True,True


In [6]:
print("Same public feature contract:", rf_features == xgb_features)
print("RF feature count:", len(rf_features))
print("XGB feature count:", len(xgb_features))
print("Official Peak-Risk threshold:", metadata["xgb"]["operational_threshold"])

Same public feature contract: True
RF feature count: 22
XGB feature count: 22
Official Peak-Risk threshold: 0.06


## Memory policy

Global built-in importance can inspect all 24 horizons sequentially.
SHAP is intentionally restricted initially to representative horizons
**h+1, h+6, h+12, h+18, h+24**, with FSA-balanced sampling.

This avoids loading all large Random Forest artifacts simultaneously.


In [7]:
print("Built-in importance horizons:", config["analysis"]["global_importance_horizons"])
print("SHAP horizons:", config["analysis"]["shap_representative_horizons"])
print("Max SHAP rows per horizon:", config["analysis"]["max_shap_rows_per_horizon"])
print("EXPL_00 RESULT: READY")


Built-in importance horizons: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
SHAP horizons: [1, 6, 12, 18, 24]
Max SHAP rows per horizon: 240
EXPL_00 RESULT: READY
